# Resonance frequency and quality factor versus temperature
This notebook shows how to model and fit the temperature dependence of a KID resonance with `citkid.res_vs_temp`, using synthetic data. Two effects are included:

- **Thermal quasiparticles (QPs)** (Mattis-Bardeen): the QP density $n_\mathrm{th}$ grows as $e^{-\Delta_0/kT}$, which lowers $f_r$ and adds loss, so both $f_r$ and $Q$ fall at high temperature.
- **Two-level systems (TLS)** in the dielectrics: as $T$ rises the TLS saturate thermally, so the TLS loss falls ($Q$ rises) and $f_r$ rises at low temperature.

With $\Delta_0 = 1.762\,k_B T_c$ and $\xi = hf_0 / 2k_BT$, the models in `citkid.res_vs_temp.funcs` are

$$\frac{f_\mathrm{qp}(T) - f_0}{f_0} = -\frac{\gamma\alpha}{4\Delta_0} S_2(T)\,\frac{n_\mathrm{th}}{N_0}, \qquad \frac{1}{Q_\mathrm{qp}(T)} = \frac{\gamma\alpha}{2\Delta_0} S_1(T)\,\frac{n_\mathrm{th}}{N_0},$$

$$\frac{f_\mathrm{TLS}(T) - f_0}{f_0} = \frac{F\delta_0}{\pi}\left[\mathrm{Re}\,\psi\!\left(\tfrac{1}{2} + \tfrac{j\xi}{\pi}\right) - \ln\tfrac{\xi}{\pi}\right], \qquad \frac{1}{Q_\mathrm{TLS}(T)} = F\delta_0 \tanh\xi,$$

and the full models add the fractional shifts and the losses:

$$f(T) = f_\mathrm{qp}(T) + f_\mathrm{TLS}(T) - f_0, \qquad \frac{1}{Q(T)} = \frac{1}{Q_\mathrm{qp}(T)} + \frac{1}{Q_\mathrm{TLS}(T)} + \delta_z.$$

| parameter | meaning |
|---|---|
| `f0` | resonance frequency at $T = 0$ (Hz) |
| `alpha` | kinetic inductance fraction $\alpha = L_k / L_\mathrm{total}$ |
| `Tc` | critical temperature (K), which sets the gap $\Delta_0 = 1.762\,k_BT_c$ |
| `Fdelta0` | TLS filling factor times the TLS loss tangent at $T = 0$ (low power) |
| `delta_z` | temperature-independent loss: $1/Q$ at $T = 0$ not from TLS (e.g. coupling, other losses) |
| `gamma` | 1, 1/2 or 1/3 for the thin-film, local or anomalous limit (not fit) |

The QP terms depend only on the product $\gamma\alpha$, so `gamma` is a fixed choice that sets how `alpha` is reported. If you fit the loaded $Q_r$, `delta_z` includes $1/Q_c$; to study internal losses, fit $Q_i$ instead.

Note: two bugs were fixed recently. `f_vs_T` now passes `gamma` to the QP term, and the sign of the QP loss in `Q_vs_T_qp` is now positive, so $Q$ falls with temperature as it should.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from citkid.res_vs_temp.funcs import (f_vs_T, Q_vs_T, f_vs_T_qp, f_vs_T_tls,
                                      Q_vs_T_qp, Q_vs_T_tls)
from citkid.res_vs_temp.fitter import (fit_f_vs_T, fit_Q_vs_T, fit_f_vs_T_tls,
                                       fit_Q_vs_T_tls)

def print_fit(names, p_true, popt, perr):
    """Print injected and fitted parameters, and the pull (fit - true) / err."""
    print(f"{'':>8} {'true':>14} {'fit':>14} {'err':>10} {'pull':>6}")
    for name, t, p, e in zip(names, p_true, popt, perr):
        pull = (p - t) / e if e > 0 else np.nan
        print(f'{name:>8} {t:14.9g} {p:14.9g} {e:10.3g} {pull:6.2f}')

## 1. The model and its components
We use parameters typical of an aluminum resonator. The QP and TLS components are also available separately: `f_vs_T_qp`, `f_vs_T_tls`, `Q_vs_T_qp` and `Q_vs_T_tls` (the last two take their own `delta_z`; here we set it to 0 to show the components alone).

In [ ]:
f0 = 500e6
alpha = 0.4
Tc = 1.3
Fdelta0 = 2e-5
delta_z = 1 / 3e5
gamma = 1

T_model = np.geomspace(0.03, 0.4, 300)
fig, axs = plt.subplots(1, 2, figsize = [10, 4], layout = 'tight')
x_tot = f_vs_T(T_model, f0, alpha, Tc, Fdelta0, gamma) / f0 - 1
x_qp = f_vs_T_qp(T_model, f0, alpha, Tc, gamma) / f0 - 1
x_tls = f_vs_T_tls(T_model, f0, Fdelta0) / f0 - 1
axs[0].plot(T_model * 1e3, x_tot * 1e6, 'k', label = 'f_vs_T')
axs[0].plot(T_model * 1e3, x_qp * 1e6, '--', label = 'f_vs_T_qp')
axs[0].plot(T_model * 1e3, x_tls * 1e6, '--', label = 'f_vs_T_tls')
axs[0].set(xlabel = 'Temperature (mK)', ylabel = r'$(f - f_0) / f_0$ (ppm)',
           ylim = [-60, 20])
axs[0].legend()

Q_tot = Q_vs_T(T_model, f0, alpha, Tc, Fdelta0, delta_z, gamma)
Q_qp = Q_vs_T_qp(T_model, f0, alpha, Tc, 0, gamma)
Q_tls = Q_vs_T_tls(T_model, f0, Fdelta0, 0)
axs[1].plot(T_model * 1e3, 1 / Q_tot, 'k', label = 'Q_vs_T')
axs[1].plot(T_model * 1e3, 1 / Q_qp, '--', label = 'Q_vs_T_qp')
axs[1].plot(T_model * 1e3, 1 / Q_tls, '--', label = 'Q_vs_T_tls')
axs[1].axhline(delta_z, color = '0.5', ls = ':', label = 'delta_z')
axs[1].set(xlabel = 'Temperature (mK)', ylabel = '1 / Q', yscale = 'log',
           ylim = [1e-6, 1e-3])
axs[1].legend()
for ax in axs:
    ax.set_xscale('log')
plt.show()

## 2. Synthetic data
We sample 30 temperatures from 40 to 380 mK and add Gaussian noise with a fixed seed: 50 Hz on $f_r$ and 1% on $Q$.

In [ ]:
rng = np.random.default_rng(0)
T = np.geomspace(0.04, 0.38, 30)
f_err = np.full(T.size, 50.)
f_data = f_vs_T(T, f0, alpha, Tc, Fdelta0, gamma)
f_data = f_data + f_err * rng.normal(size = T.size)
Q_true = Q_vs_T(T, f0, alpha, Tc, Fdelta0, delta_z, gamma)
Q_err = 0.01 * Q_true
Q_data = Q_true + Q_err * rng.normal(size = T.size)

fig, axs = plt.subplots(1, 2, figsize = [10, 4], layout = 'tight')
axs[0].errorbar(T * 1e3, (f_data - f0) / 1e3, f_err / 1e3, fmt = '.')
axs[0].set(xlabel = 'Temperature (mK)', ylabel = r'$f_r - f_0$ (kHz)')
axs[1].errorbar(T * 1e3, Q_data / 1e3, Q_err / 1e3, fmt = '.')
axs[1].set(xlabel = 'Temperature (mK)', ylabel = r'$Q / 1000$')
plt.show()

## 3. Fitting $f_r(T)$
`fit_f_vs_T(T, f, gamma, Tc_guess, f_err = ...)` fits `[f0, alpha, Tc, Fdelta0]`. The initial guess (`citkid.res_vs_temp.guess.guess_p0_f_vs_T`) takes `f0` and `Fdelta0` from a TLS fit to the colder half of the data, `alpha = 0.7 / gamma`, and `Tc = Tc_guess`. The default bounds keep `f0` within $10^{-5}$ of its guess, `alpha` and `Tc` within a factor 3 and 2, and `Fdelta0` within a factor 3. It returns `p0, popt, perr, (fig, ax)`.

Pass the measurement errors (`f_err`) if you have them. Without them, points are weighted equally and the parameter errors are scaled by the fit residuals.

The fit can stop in a local minimum depending on `Tc_guess`. A simple remedy is to try a few values of `Tc_guess` and keep the fit with the lowest $\chi^2$:

In [ ]:
def chi2_f(popt):
    """Chi-squared of an f_vs_T fit to the data."""
    return np.sum(((f_data - f_vs_T(T, *popt, gamma = gamma)) / f_err) ** 2)

fits = []
for Tc_guess in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    p0, popt, perr, _ = fit_f_vs_T(T, f_data, gamma = gamma,
                                   Tc_guess = Tc_guess, f_err = f_err,
                                   catch_exceptions = True)
    fits.append((chi2_f(popt), Tc_guess, popt, perr))
    print(f'Tc_guess = {Tc_guess:.1f} K: chi2 = {fits[-1][0]:10.1f}, '
          f'alpha = {popt[1]:.3f}, Tc = {popt[2]:.3f} K')
chi2, Tc_guess_best, popt_f, perr_f = min(fits, key = lambda fit: fit[0])
print(f'\nbest: Tc_guess = {Tc_guess_best}, chi2 = {chi2:.1f} '
      f'for {T.size - 4} degrees of freedom\n')
print_fit(['f0', 'alpha', 'Tc', 'Fdelta0'], [f0, alpha, Tc, Fdelta0],
          popt_f, perr_f)

`plotq = True` plots the data, fit and initial guess (`citkid.res_vs_temp.plot.plot_f_vs_T`).

In [ ]:
p0, popt, perr, (fig, ax) = fit_f_vs_T(T, f_data, gamma = gamma,
                                       Tc_guess = Tc_guess_best,
                                       f_err = f_err, plotq = True)
plt.show()

## 4. Fitting $Q(T)$
`fit_Q_vs_T(T, Q, f0_guess, gamma, Tc_guess, Q_err = ...)` fits `[f0, alpha, Tc, Fdelta0, delta_z]`. $Q$ depends on `f0` only through $hf_0/k_BT$, so `f0` is not constrained by $Q$ data: pass the value from the $f_r$ fit as `f0_guess` and ignore the fitted `f0` and its error. Again we scan `Tc_guess`.

In [ ]:
def chi2_Q(popt):
    """Chi-squared of a Q_vs_T fit to the data."""
    return np.sum(((Q_data - Q_vs_T(T, *popt, gamma = gamma)) / Q_err) ** 2)

fits = []
for Tc_guess in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    p0, popt, perr, _ = fit_Q_vs_T(T, Q_data, popt_f[0], gamma = gamma,
                                   Tc_guess = Tc_guess, Q_err = Q_err,
                                   catch_exceptions = True)
    fits.append((chi2_Q(popt), Tc_guess, popt, perr))
    print(f'Tc_guess = {Tc_guess:.1f} K: chi2 = {fits[-1][0]:10.1f}, '
          f'alpha = {popt[1]:.3f}, Tc = {popt[2]:.3f} K')
chi2, Tc_guess_best_Q, popt_Q, perr_Q = min(fits, key = lambda fit: fit[0])
print(f'\nbest: Tc_guess = {Tc_guess_best_Q}, chi2 = {chi2:.1f} '
      f'for {T.size - 4} degrees of freedom\n')
print_fit(['alpha', 'Tc', 'Fdelta0', 'delta_z'],
          [alpha, Tc, Fdelta0, delta_z], popt_Q[1:], perr_Q[1:])

`Fdelta0` and `delta_z` are anticorrelated in the $Q$ fit (both add loss at low temperature), so their pulls tend to have opposite signs. The guess (dotted) uses `delta_z = 1 / min(Q)`, which is far off when the minimum $Q$ is set by QPs at the highest temperature, which is one reason the `Tc_guess` scan above is needed.

In [ ]:
p0, popt, perr, (fig, ax) = fit_Q_vs_T(T, Q_data, popt_f[0], gamma = gamma,
                                       Tc_guess = Tc_guess_best_Q,
                                       Q_err = Q_err, plotq = True)
plt.show()

## 5. Fit options
- `enforced_alpha`: hold `alpha` fixed (e.g. from a simulation or from the $f_r$ fit). It is returned in `popt` with a NaN error. This also removes the strong `alpha`-`Tc` correlation.
- `guess`, `bounds`: replace the automatic initial guess or bounds (same parameter order as `popt`).
- `return_dataframe = True`: return a `pd.Series` with the guess, fit and errors instead of the lists, for building a table of many resonators (`citkid.res_vs_temp.data_io`).

Here we fit $Q(T)$ with `alpha` fixed to the value from the $f_r$ fit:

In [ ]:
row, _ = fit_Q_vs_T(T, Q_data, popt_f[0], gamma = gamma,
                    Tc_guess = Tc_guess_best_Q, Q_err = Q_err,
                    enforced_alpha = popt_f[1], return_dataframe = True)
row

## 6. TLS-only fits at low temperature
Well below $T_c$ the QP terms vanish, and the TLS components alone describe the data. `fit_f_vs_T_tls` (parameters `[f0, Fdelta0]`) and `fit_Q_vs_T_tls` (`[f0, Fdelta0, delta_z]`) fit them. The QP fitters `fit_f_vs_T_qp` and `fit_Q_vs_T_qp` are the counterparts for data where TLS effects are negligible. For this resonator the QP frequency shift is already several ppm at 200 mK, so we use only the points below 130 mK.

In [ ]:
cold = T < 0.13
p0, popt, perr, (fig, ax) = fit_f_vs_T_tls(T[cold], f_data[cold],
                                           f_err = f_err[cold], plotq = True)
plt.show()
print_fit(['f0', 'Fdelta0'], [f0, Fdelta0], popt, perr)

p0, popt, perr, _ = fit_Q_vs_T_tls(T[cold], Q_data[cold], popt[0],
                                   Q_err = Q_err[cold])
print()
print_fit(['Fdelta0', 'delta_z'], [Fdelta0, delta_z], popt[1:], perr[1:])